# Day 01 — Getting Started with LLMs using Google Gemini

Today’s learning: calling a Large Language Model (LLM) from Python and building a small recipe-generation function.

## What you will learn
1. What an LLM does (at a practical level).
2. How to install the Gemini Python SDK and `python-dotenv`.
3. How to keep an API key in an environment file.
4. How to initialize the Gemini client.
5. How to write a prompt using Python f-strings.
6. How to create a reusable function that sends a prompt to Gemini and returns its response.

> **Security:** Never commit your real API key or `.env` file to GitHub. This notebook reads `GEMINI_API_KEY` from your environment.

## 1. Install the required packages

Run this cell in Jupyter. The `!` tells the notebook to run a terminal command.

- `google-genai`: Google's official Python SDK for Gemini.
- `python-dotenv`: loads values from a local `.env` file into environment variables.

If you are using a different Python environment, make sure Jupyter is running with that same environment.

In [ ]:
%pip install -q google-genai python-dotenv

## 2. Store your Gemini API key safely

Create a file named **`.env`** in the project root (the same level as this notebook's parent project folder) with this format:

```text
GEMINI_API_KEY=put_your_real_key_here
```

Do not add quotes unless the key itself requires them. Do not paste the real key into the notebook.

For this repository, `.gitignore` excludes `.env`. If your local file is still named `.env.txt`, either rename it to `.env` or pass its exact path to `load_dotenv()`.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
from google import genai

# This notebook is in notebooks/, so the project root is its parent folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

env_path = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=env_path)

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(
        f"GEMINI_API_KEY was not found. Create a .env file at: {env_path}"
    )

client = genai.Client(api_key=api_key)
print("Gemini client initialized successfully!")

## 3. Generate a simple LLM response

An LLM receives a prompt (instructions and context) and generates a response. The model name below is an example; use a model currently available to your Gemini API project.

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain a Large Language Model in simple English in 3 bullet points."
)

print(response.text)

## 4. Build a recipe generator

We will create a reusable Python function with three inputs:

- `ingredients`: a list of ingredients, such as `["tomato", "onion", "chilli"]`
- `cuisine`: for example, `"Indian"`, `"Italian"` or `"Any"`
- `diet`: for example, `"vegetarian"`, `"vegan"`, `"Jain"`, `"non-vegetarian"` or `"Any"`

### Important Python idea: `join()`

Use `", ".join(ingredients)` to turn a **list** into readable text. For example:

```python
", ".join(["tomato", "onion", "chilli"])
# 'tomato, onion, chilli'
```

Calling `','.join()` on a single string joins its individual characters, which is not what we want here.

In [ ]:
def generate_recipe(ingredients: list[str], cuisine: str, diet: str) -> str:
    """Generate a short recipe from ingredients using Gemini."""

    ingredient_text = ", ".join(ingredients)

    prompt = f"""
    Create one practical food recipe using these ingredients: {ingredient_text}

    Cuisine: {cuisine}
    Dietary preference: {diet}

    Keep the recipe under 100 words.
    Mention the dish name and give concise preparation steps.
    Respect the dietary preference. If the ingredients conflict with it,
    explain that briefly instead of suggesting an unsuitable recipe.
    """

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        if not response.text:
            return "The model returned an empty response. Please try again."

        return response.text

    except Exception as error:
        return f"Sorry, recipe generation failed: {error}" 

## 5. Test the function

Change the ingredients, cuisine, and dietary preference to experiment with different prompts.

In [ ]:
recipe = generate_recipe(
    ingredients=["tomato", "onion", "chilli", "oil", "garlic paste", "chicken"],
    cuisine="Indian",
    diet="non-vegetarian"
)

print(recipe)

## 6. What happened? (learning notes)

1. Python imported the Gemini SDK and dotenv helper.
2. `load_dotenv()` loaded the API key from `.env`.
3. `genai.Client(api_key=api_key)` created the Gemini client.
4. The function combined the inputs into a prompt using an f-string.
5. `client.models.generate_content()` sent the prompt to Gemini.
6. `response.text` contained the generated answer.

### Key terms
- **LLM:** Large Language Model; a model that generates or transforms text based on input.
- **Prompt:** The instructions and context given to the model.
- **API:** A way for one program to request a service from another.
- **SDK:** A library that makes it easier to use an API from a programming language.
- **Environment variable:** A value stored outside source code, often used for configuration and secrets.
- **Client:** The SDK object used to make requests to the Gemini API.

### Troubleshooting
- `NameError: client is not defined`: run the client-initialization cell before calling the function.
- API key not found: check the `.env` filename, its location, and that the variable is named `GEMINI_API_KEY`.
- Package import error: run the installation cell in the same Jupyter environment, then restart the kernel if needed.
- Model error: check the current model IDs and availability for your Gemini API account.

### Practice tasks
1. Generate an Italian recipe using three ingredients.
2. Try a vegan recipe and remove animal-derived ingredients.
3. Add a fourth parameter called `max_words`.
4. Ask Gemini to return the recipe with headings: Ingredients and Steps.
5. Add input validation for an empty ingredient list.